# Tokenization in Language Models

Token are the fundamental unit or atoms of the large language models
- Context size for inputs to language models are defined in terms of tokens. It defines the maximum number of tokens that a model can see for prior context.
- The vocabulary is the full set of tokens.
- The last head layer in a language model provides the probability of each token being the next item in the sequence.
- Tokenization is the process of converting input string to tokens and vice versa for outputs (from tokens to strings)

The process consists of taking string input, segmenting it appropriately, translating the sequence segments to appropriate tokens, the token are used to lookup the embedding and the token embedding are fed as input to the model

Tokenization is important and at the heart of good functioning LLMs. Issues in tokenization can cause wierd bugs.

Tokenization can be a cause for many concerns with LLM, like: 
LLM cannot spell words or cannot do super simple string processing or super simple arithmetic. GPT2 had trouble coding in python, is not good with JSON (prefers YAML). LLM used to break if asked about SolidGoldMagikarp, weird warning with 'trailing whitespce', halts when it sees "<|endoftext|>"

LLMs like GPT are worse at non-English languages (e.g. Japanese) as the training set is larger for English language than non-English. Same is true for Tokenizers - the training set in English is larger.
As a result the non-English languages has lot more tokens to represent the same sentence than English. In English the tokens are longer sequence of characters and non-English has short sequence of characters. From the LLM's input perspective - non-English takes a lot more tokens to represent the same sentence than English; the sentence is stretched over many tokens, the transformer runs out of context length and gets to see a shorter sequence to predict the next token which impacts its performance. When tokens are denser, there is larger sequences passed to the LLM within the same token count, it gets to see larger sentences as context to predict what comes next. GPT4 densified python tokens to be able to attend to more code within the context length; this improved python coding ability in GPT4 vs GPT2.

Larger number of characters in tokens is not always good; it increases the vocabulary size, which leads to incerased embedding matrix size, as well as the layer predicting next token needs to evaluate probabilities over larger set of tokens. There is a trade off between sequence length (dense) tokens to number of tokens in vocabulary. A good balance between the two for a good LLM. 

Language Foundation models use string inputs as data. Strings are immutable sequences of unicode code points. LLMs use Byte Pair Encoding (BPE) Algorithm to define tokens. Titoken library from Open AI and sentence piece used in Llama and Mistral series of models implement BPE. In this notebook we implement byte pair encoding from first principles. We also understand the difference between the implementation in tiktoken and sentence piece implementat


Reference: 
- Language Models are Unsupervised Mutlitask Learners Section 2.2
https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf
- Llama2 Open foundation and fine-tuned chat models https://arxiv.org/abs/2307.09288
- Andrej Karapathy https://www.youtube.com/watch?v=zduSFxRajkE&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=9
- Tokenizer App: https://tiktokenizer.vercel.app
- Introduction to Unicode: https://www.reedbeta.com/blog/programmers-intro-to-unicode/
- Byte Pair Algorithm: https://en.wikipedia.org/wiki/Byte-pair_encoding
- Megabyte - Multiscale transformers: https://arxiv.org/pdf/2305.07185
- Learning to compress prompts with Gist tokens https://arxiv.org/abs/2304.08467

### Unicode Code Points
In python strings are immutable sequences of unicode code points. Unicode code points are defined by Unicode Consortium as a part of Unicode Standard. They define what the characters look like and what integers represent them. There are 3 standards: a) Unicode8: Provides 1 to 4 bytes variable length encoding b) Unicode 16 and 32 provides fixed length encoding and longer bytes

UTF-8 is the only encoding that is backward compatible to ASCII. While Unicode8 provides variable length compact encoding, it is only 256 vocab size. Unicode16 and 32 provide much larger size encoding, but are wasteful in the space they occupy due to fixed length. Also, note that Unicode code points are alive and can change as the standards are changed.
Refer Introduction to Unicode: https://www.reedbeta.com/blog/programmers-intro-to-unicode/

### Byte Pair Algorithm

##### Why?
Fine grained vocabulary at character level while it keeps the embedding first layer and last layer small, it has the disadvantage that the input representation is very rarified and the transformer architecture will only be able to see limited prior text based on context length that it maintains for computation. In a case where we are able to create tokens with multi character length the input representation is made denser, the transformer architecture within the same context length is able to see longer text sequence to inform its prediction. That does increase the vocabulary size.

Byte Pair Algorithm provides a method to create tokens with longer character sequences - in other words compress more sequence into the token.

##### Are there alternatives?
If we are to represent inputs in terms of its atomic bytes then we will have very large context legnths and the architecture to handle this will have to different the normal tranformers; Megabyte paper proposed a hierarchical struture to achieve this will localized pathces unified by global model, but this yet to be proven at production scale.


##### How?
It iteratively does the following: a) identify the most common contiguous pair of characters b) Mint a new token to represent that pair of characters c) Replace the pair of characters with the new token d) Repeat steps a to c number of times - a hyperparameter, based on the new vocabulary size needed.
This process is carried out on a training dataset

Refer: Byte Pair Algorithm https://en.wikipedia.org/wiki/Byte-pair_encoding

In [31]:
#
print(ord('a')) # unicode code point of 'a'
print(chr(100)) # unicode integer to character
print(chr(345))
print(chr(1150))
print("Hello".encode("utf-16"))
print("Hello".encode("utf-8").hex())
print(list("Hello".encode("utf-8")))
print(list("Hello".encode("utf-32"))) # notice a many zeros that are listed for ascii characters specifically due to fixed length

97
d
ř
Ѿ
b'\xff\xfeH\x00e\x00l\x00l\x00o\x00'
48656c6c6f
[72, 101, 108, 108, 111]
[255, 254, 0, 0, 72, 0, 0, 0, 101, 0, 0, 0, 108, 0, 0, 0, 108, 0, 0, 0, 111, 0, 0, 0]


In [28]:
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
token_bytes = text.encode("utf-8") # raw bytes
vocab_set = {i: chr(i) for i in set(map(int, token_bytes))} # uses function int to convert bytes to integer representation
print("Vocabulary", vocab_set)
tokens = list(map(int, token_bytes)) # map ---> maps integers to bytes
print(f"-----Text - length {len(text)}------")
print(text)
print(f"-----Tokens - length {len(tokens)}----ascii characters are one byte, special characters are upto 4 bytes (variable length encoding in utf-8)---")
print(tokens)

Vocabulary {128: '\x80', 131: '\x83', 132: '\x84', 133: '\x85', 135: '\x87', 137: '\x89', 140: '\x8c', 142: '\x8e', 143: '\x8f', 146: '\x92', 147: '\x93', 148: '\x94', 152: '\x98', 153: '\x99', 156: '\x9c', 157: '\x9d', 158: '\x9e', 159: '\x9f', 32: ' ', 33: '!', 164: '¤', 168: '¨', 169: '©', 170: 'ª', 40: '(', 44: ',', 41: ')', 174: '®', 46: '.', 45: '-', 48: '0', 179: '³', 180: '´', 181: 'µ', 51: '3', 186: 'º', 188: '¼', 189: '½', 63: '?', 66: 'B', 73: 'I', 83: 'S', 84: 'T', 85: 'U', 87: 'W', 120: 'x', 95: '_', 97: 'a', 226: 'â', 99: 'c', 100: 'd', 101: 'e', 102: 'f', 103: 'g', 104: 'h', 105: 'i', 98: 'b', 107: 'k', 108: 'l', 109: 'm', 110: 'n', 239: 'ï', 240: 'ð', 111: 'o', 114: 'r', 115: 's', 116: 't', 112: 'p', 118: 'v', 119: 'w', 117: 'u', 121: 'y', 122: 'z'}
-----Text - length 533------
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like

In [29]:
def get_stats(id_list):
    count_dict = {}
    for tok1, tok2 in zip(id_list, id_list[1:]):
        count_dict[(tok1, tok2)] = count_dict.get((tok1, tok2), 0) + 1
        count_dict_sorted = sorted(count_dict, key=lambda x: count_dict[x])
        #count_dict_sorted = sorted(((v, k) for k, v in count_dict.items()), reverse=True) alternate method to sort
        top_pair = count_dict_sorted[-1]
        # top_pair = max(count_dict, key=count_dict.get) # alternate method to get top pair
        top_pair_chr = (chr(top_pair[0]), chr(top_pair[1]))
    return count_dict, top_pair, top_pair_chr

count_dict, top_pair, top_pair_chr = get_stats(tokens)
print(count_dict)
print(count_dict[top_pair], top_pair, top_pair_chr)
print(max(vocab_set))

def add_to_vocab(vocab_set, top_pair):
    mint_token_id = max(vocab_set) + 1
    top_pair_text = vocab_set[top_pair[0]] + vocab_set[top_pair[1]]
    vocab_set[mint_token] = top_pair_text
    print(mint_token, vocab_set[mint_token], top_pair)
    print("----------new vocab_set----------", vocab_set)
    return vocab_set, mint_token_id

vocab_set, new_minted_id = add_to_vocab(vocab_set, top_pair)

def merge(id_list, top_pair, new_minted_id):
    new_tokens = []
    i = 0
    while i < len(tokens):
        if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == top_pair:
            #print(tokens[:i+2])
            new_tokens.append(new_minted_id)
            #print(new_tokens)
            i += 2
        else:
            new_tokens.append(tokens[i])
            i += 1

    return new_tokens

new_tokens = merge(tokens, top_pair, new_minted_id)
print(f"---------Tokens - length {len(new_tokens)}---------")
print("".join(vocab_set[i] for i in new_tokens))

{(239, 188): 1, (188, 181): 1, (181, 239): 1, (239, 189): 6, (189, 142): 1, (142, 239): 1, (189, 137): 1, (137, 239): 1, (189, 131): 1, (131, 239): 1, (189, 143): 1, (143, 239): 1, (189, 132): 1, (132, 239): 1, (189, 133): 1, (133, 33): 1, (33, 32): 2, (32, 240): 3, (240, 159): 15, (159, 133): 7, (133, 164): 1, (164, 240): 1, (133, 157): 1, (157, 240): 1, (133, 152): 1, (152, 240): 1, (133, 146): 1, (146, 240): 1, (133, 158): 1, (158, 240): 1, (133, 147): 1, (147, 240): 1, (133, 148): 1, (148, 226): 1, (226, 128): 12, (128, 189): 1, (189, 32): 1, (159, 135): 7, (135, 186): 1, (186, 226): 1, (128, 140): 6, (140, 240): 6, (135, 179): 1, (179, 226): 1, (135, 174): 1, (174, 226): 1, (135, 168): 1, (168, 226): 1, (135, 180): 1, (180, 226): 1, (135, 169): 1, (169, 226): 1, (135, 170): 1, (170, 33): 1, (159, 152): 1, (152, 132): 1, (132, 32): 1, (32, 84): 1, (84, 104): 1, (104, 101): 6, (101, 32): 20, (32, 118): 1, (118, 101): 3, (101, 114): 6, (114, 121): 2, (121, 32): 2, (32, 110): 2, (110,

### SUMMARY ---> Tokenization - Byte Pair Encoding Algorithm

In [61]:
# Putting it together

def get_stats(id_list):
    count_dict = {}
    for tok1, tok2 in zip(id_list, id_list[1:]):
        count_dict[(tok1, tok2)] = count_dict.get((tok1, tok2), 0) + 1
        count_dict_sorted = sorted(count_dict, key=lambda x: count_dict[x])
        #count_dict_sorted = sorted(((v, k) for k, v in count_dict.items()), reverse=True) alternate method to sort
        top_pair = count_dict_sorted[-1]
        # top_pair = max(count_dict, key=count_dict.get) # alternate method to get top pair
        top_pair_chr = (chr(top_pair[0]), chr(top_pair[1]))
    return count_dict, top_pair, top_pair_chr

def add_to_vocab(v_set, top_pair):
    mint_token_id = max(v_set) + 1
    top_pair_text = v_set[top_pair[0]] + v_set[top_pair[1]]
    v_set[mint_token_id] = top_pair_text
    return v_set, mint_token_id

def merge(id_list, top_pair, new_minted_id):
    new_tokens = []
    i = 0
    while i < len(id_list):
        if i < len(id_list)-1 and (id_list[i], id_list[i+1]) == top_pair:
            #print(tokens[:i+2])
            new_tokens.append(new_minted_id)
            #print(new_tokens)
            i += 2
        else:
            new_tokens.append(id_list[i])
            i += 1

    return new_tokens

    
with open('Unicode.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print('length of input - number of characters', len(text))
print('initial 15 characters in input: ', text[:15])

token_bytes = text.encode('utf-8')
tokens = list(map(int, token_bytes)) # map ---> maps integers to bytes
vocab = {ix: bytes([ix]) for ix in range(256)} # uses function int to convert bytes to integer representation
print("Vocabulary", vocab_set)
print(f"-----Text - length {len(text)}------")
print(f"-----Tokens - length {len(tokens)}----")
print("---ascii characters are one byte, special characters are upto 4 bytes (variable length encoding in utf-8)---")

final_vocab_size = 276
num_iters = final_vocab_size - len(vocab) # hyperparameter
ids = list(tokens) # a copy operation
merges = {}
for j in range(num_iters):
    count_dict, top_p, top_pair_chr = get_stats(ids)
    #print("count_dict", count_dict)
    print("top_pair", count_dict[top_p], top_p, top_pair_chr)
    
    vocab, new_minted_id = add_to_vocab(vocab, top_p)
    #print("----------new vocab_set----------", v_set)
    print("new_minted_id", new_minted_id, vocab[new_minted_id])
    ids = merge(ids, top_p, new_minted_id)
    merges[top_p] = new_minted_id
    print(f"---------Tokens - length {len(ids)}---------")
    print(f"merging {top_p} inot a new token {new_minted_id}")
    #print("".join(vocab_set[i] for i in new_tokens))

# this is tokenizatio training

length of input - number of characters 23029
initial 15 characters in input:  A Programmer’s 
Vocabulary {65: 'A', 32: ' ', 80: 'P', 114: 'r', 111: 'o', 103: 'g', 97: 'a', 109: 'm', 101: 'e', 226: 'â', 128: '\x80', 153: '\x99', 115: 's', 73: 'I', 110: 'n', 116: 't', 100: 'd', 117: 'u', 99: 'c', 105: 'i', 85: 'U', 10: '\n', 77: 'M', 104: 'h', 51: '3', 44: ',', 50: '2', 48: '0', 49: '1', 55: '7', 194: 'Â', 183: '·', 67: 'C', 53: '5', 239: 'ï', 188: '¼', 181: 'µ', 189: '½', 142: '\x8e', 137: '\x89', 131: '\x83', 143: '\x8f', 132: '\x84', 133: '\x85', 33: '!', 240: 'ð', 159: '\x9f', 164: '¤', 157: '\x9d', 152: '\x98', 146: '\x92', 158: '\x9e', 147: '\x93', 148: '\x94', 135: '\x87', 186: 'º', 140: '\x8c', 179: '³', 174: '®', 168: '¨', 180: '´', 169: '©', 170: 'ª', 84: 'T', 118: 'v', 121: 'y', 107: 'k', 102: 'f', 119: 'w', 112: 'p', 108: 'l', 46: '.', 87: 'W', 156: '\x9c', 40: '(', 95: '_', 63: '?', 41: ')', 66: 'B', 98: 'b', 45: '-', 83: 'S', 122: 'z', 120: 'x', 72: 'H', 47: '/', 68: 'D', 7

### Given merges after training Encode string--->tokens and Decode token--->string


In [91]:
# Decoding
# Given a sequence of integers in range [0, vocab_size], what is the text?
# note that UTF8 is multibyte. 
# When predicting or reading bytes we can come across Partial Data (like in Streaming)
# When reading from a network or serial port, you might have received only half of a multi-byte character, 
# leading to a decoding error on the next read. Use errors="replace" or "ignore": 
# We don't need to read every character perfectly, (not errors = "strict" default)
# we can bypass the error by replacing invalid characters with a marker (?).
def decode(ids):
    # given id (list of integers), return string
    tokens = (b"".join(vocab[i] for i in ids)) # raw bytes
    text = tokens.decode("utf-8", errors="replace") # throws an error as the start byte does not follow unicode rules
    return text

def encode(text):
    # given a string return list of integers (tokens)
    bytes_seq = list(text.encode("utf-8")) 
    tokens = list(bytes_seq)
    '''##### option1: method start 
    for k, v in merges.items(): # in Python3.7 and later, dict items are guaranteed to be sorted by insert order
        tokens = merge(tokens, k, v)
    ##### option1: method end'''
    ##### option2: method start 
    while len(tokens) >= 2: # otherwise decode will not work for one char strings
        stats, _, _ = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    ##### option2: method end
    return tokens

## test cases
text = 'A Programmer’s Introduction to Unicode\nMarch 3, 2017 · Coding · 25 Comments\n\nＵｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺\u200c🇳\u200c🇮\u200c🇨\u200c🇴\u200c🇩\u200c🇪! 😄'
text2 = decode(encode(text))
print(text == text2)
text1 = 'Hello World!'
text12 = decode(encode(text1))
print(text1 == text12)

True
True


### Adding rules to BPE tokenziation
In GPT2 paper; they observed that common words like 'dog.' 'dog?' 'dog!' all form different tokens. So GPT2 enforces rules that some special characters can be combined with words like spaces but not all special characters i.e., rules to combine alpha characters to punctuation characters. This is done using regex patterns
 (python regex is an extension on python re module)
- GPT2 Section 2.2
https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf

The regex separates the full text to a list of shorter texts and processes them independently within and the result token of the processing are concatenated. Thus avoiding merges across character classes. It has also additional rules on top of chunking and BPE. Code was never released for training. Only inference (encode / decode) is released. Tiktoken is openAI library it has released for tokenization (inference) not training. It gives GPT2 or GPT4 tokens. White spaces are merged in GPT4 not in GPT2. GPT4 merges numbers upto 3digits only to prevent tokens that are long number sequences (vs GPT2). GPT2 vocabulary size from ~50K when to ~100K in GPT4.

The equivalent of merges dict in GPT2 is stored in vocab.bpe. The equivalent of our vocab is stored in encoder.json on GPT2. These two files in GPT2 are necessary to use their encoder.py or tiktoken library

##### Special Tokens
GPT2 has 50K merges, 256 unicode 8 bytes, 1 special token <|endoftext|>. It is end of document token inserted between two documents to let the transformer know the next document is unrelated to the previous one. It has to learn from data whatever is necessary to wipe off from memory what came before and know that what comes next is unrelated to what came before.

Special tokens like end of text etc are created by GTP2, user can create new ones and register them when using tiktoken library. There is code in encoder to handle these special texts and replace them with defined token numbers before putting them through the vocab or merges logic for encoding. They are typically outside of byte pair encoding. These special tokens concepts are also used in finetuning (<|im_start|>, <|im_end|> and chatGPT to delimit conversations for example.

When special tokens are added their embeddings need to be learned and the language head - final layer predicting probabilities of tokens need to also consider this token for predictions

In [100]:
import regex as re
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
print(re.findall(gpt2pat, "Hello World how's you've been? 123#$^^$% HOW'S      got     "))
# capitalization was not dealt with in GPT2, it has been dealt with in GPT higher releases

['Hello', ' World', ' how', "'s", ' you', "'ve", ' been', '?', ' 123', '#$^^$%', ' HOW', "'", 'S', '     ', ' got', '     ']


In [57]:
# Code to observe tiktoken GPT2 encoder decoder files
# to download use
#!wget https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/vocab.bpe
#!wget https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/encoder.json

# In GPT2 encoder.py there is encoder and decoder - algorithm is same as above. 
# In addition there is byte encoder and byte decoder like what we attempted to - but is not necessary

184

### Sentence Piece Google library
Algorithm defined by google, commonly used in LMs. It can efficiently both train and infer tokenizers. It support multiple algorithms for tokenization, one of them is BPE. It is used in both Llama and Mistral series of models.

- Tiktoken encodes utf-8 bytes and then BPEs the bytes
- Sentence BPEs the code points and optionally falls back to utf8-bytes for rare code points (rarity is determined by character coverage hyperparameter), which then get translated to byte tokens

The big difference is that sentencepiece runs BPE on unicode code points directly. For rare codepoints that appear very few times in the text, it either maps them to UNK token or if byte_fallback is turned on, it encode them with utf-8 and then encodes the raw bytes.

Reference: Github: https://github.com/google/sentencepiece
Sentencepiece paper: https://arxiv.org/abs/1808.06226
Training options: https://github.com/google/sentencepiece/doc/options.md (many options are irrelvant to BPE)
proto buf: https://github.com/google/sentencepiece/src/sentencepiece_model.proto

In [5]:
import sentencepiece as spm
# write a toy.txt file with some random text
with open("toy.txt", "w", encoding="utf-8") as f:
    f.write(''.join(["SentencePiece is an unsupervised text tokenizer and detokenizer mainly",
            "for Neural Network-based text generation systems where the vocabulary size is predetermined prior to",
            "the neural model training. SentencePiece implements subword units (e.g., byte-pair-encoding (BPE) ",
            "[Sennrich et al.] and unigram language model [Kudo.]) with the extension of direct training from raw ",
            "sentences. SentencePiece allows us to make a purely end-to-end system that does not depend on",
             "language-specific pre/postprocessing"]))
# train a sentencepiece model on it
# the settings here are (best effort) those used for training Llama2
# There is longer history to this library. It treats sentences as individual training examples
# There is always a definition level discussion of what is a sentence, what are the concepts with other language
# Prefer to use the file as giant stream of bytes. For deep learning do not touch the data as far as possible
# There is word/NLP processing like root word etc not preferred for LM
import os
options = dict(
    # input spec
    input="toy.txt",
    input_format="text",
    # output spec
    model_prefix="tok400", # output filename prefix
    # algorithm spec
    # BPE alg
    model_type="bpe",
    vocab_size=400,
    # normalization - this was prevelant in NLP like root words, word processing. Not preferred in LM
    normalization_rule_name="identity", # ew, turn off normalization
    remove_extra_whitespaces=False,
    input_sentence_size=200000000, # max number of training sentences
    max_sentence_length=4192, # max number of bytes per sentence
    seed_sentencepiece_size=1000000,
    shuffle_input_sentence=True,
    # rare word treatment / rare code points
    character_coverage=0.99995,
    byte_fallback=True,
    # merge rules # similar to tiktoken using regex expressions to split by character category
    split_digits=True,
    split_by_unicode_script=True,
    split_by_whitespace=True,
    max_sentencepiece_length=16,
    add_dummy_prefix=True, # adds extra space in front of words that do not have it
    allow_whitespace_only_pieces=True,
    # special tokens
    unk_id=0, # the UNK token MUST exist
    bos_id=1, # the others are optional, set to -1 to turn off
    eos_id=2,
    pad_id=-1,
    # systems
    num_threads=os.cpu_count(), # use ~all system resources
)

spm.SentencePieceTrainer.train(**options)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: toy.txt
  input_format: text
  model_prefix: tok400
  model_type: BPE
  vocab_size: 400
  self_test_sample_size: 0
  character_coverage: 0.99995
  input_sentence_size: 200000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 4
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 1
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  diffe

In [7]:
sp = spm.SentencePieceProcessor()
sp.load('tok400.model')
vocab = [[sp.id_to_piece(idx), idx] for idx in range(sp.get_piece_size())]
vocab 
# note since byte fallback is True it uses 256 unicode bytes and tokenizes them 
# and then identifies code point merges BPE. If that flag is set to false the 256 unicode bytes are not listed
# a lot more codepoint merges are identified to occupy the vocabulary capacity
# special tokens, byte tokens, merge tokens, individual code points.
# Merge tokens and code point token are based on those that occur in training set

[['<unk>', 0],
 ['<s>', 1],
 ['</s>', 2],
 ['<0x00>', 3],
 ['<0x01>', 4],
 ['<0x02>', 5],
 ['<0x03>', 6],
 ['<0x04>', 7],
 ['<0x05>', 8],
 ['<0x06>', 9],
 ['<0x07>', 10],
 ['<0x08>', 11],
 ['<0x09>', 12],
 ['<0x0A>', 13],
 ['<0x0B>', 14],
 ['<0x0C>', 15],
 ['<0x0D>', 16],
 ['<0x0E>', 17],
 ['<0x0F>', 18],
 ['<0x10>', 19],
 ['<0x11>', 20],
 ['<0x12>', 21],
 ['<0x13>', 22],
 ['<0x14>', 23],
 ['<0x15>', 24],
 ['<0x16>', 25],
 ['<0x17>', 26],
 ['<0x18>', 27],
 ['<0x19>', 28],
 ['<0x1A>', 29],
 ['<0x1B>', 30],
 ['<0x1C>', 31],
 ['<0x1D>', 32],
 ['<0x1E>', 33],
 ['<0x1F>', 34],
 ['<0x20>', 35],
 ['<0x21>', 36],
 ['<0x22>', 37],
 ['<0x23>', 38],
 ['<0x24>', 39],
 ['<0x25>', 40],
 ['<0x26>', 41],
 ['<0x27>', 42],
 ['<0x28>', 43],
 ['<0x29>', 44],
 ['<0x2A>', 45],
 ['<0x2B>', 46],
 ['<0x2C>', 47],
 ['<0x2D>', 48],
 ['<0x2E>', 49],
 ['<0x2F>', 50],
 ['<0x30>', 51],
 ['<0x31>', 52],
 ['<0x32>', 53],
 ['<0x33>', 54],
 ['<0x34>', 55],
 ['<0x35>', 56],
 ['<0x36>', 57],
 ['<0x37>', 58],
 ['<0x38>', 5

1. note '!' is not seen in training is unknown token or converted to bytes based on flag chosen if byte fallback is False we have lot more merges and unknown token is <unk> 
2. For a LM how does it train when the data has too many unk tokens. It is not a preferred property. Prefer to keep byte fallback flag true
3. To identify ' world' and 'world' as same word, add_dummy_prefix=True. Below you see '_' added before Hello. This is different from titokenizer where the LM has to learn that ' world' and 'world' mean the same


In [8]:
ids = sp.encode("Hello World!")
print(ids)
print([sp.id_to_piece(idx) for idx in ids])
# 1. note '!' is not seen in training is unknown token or converted to bytes based on flag chosen
# if byte fallback is False we have lot more merges and unknown token is <unk> 
# 2. For a LM how does it train when the data has too many unk tokens. It is not a preferred property
# Prefer to keep byte fallback flag true
# 3. To identify ' world' and 'world' as same word, add_dummy_prefix=True. Below you see '_' added before Hello
# This is different from titokenizer where the LM has to learn that ' world' and 'world' mean the same

[362, 75, 361, 372, 356, 362, 90, 269, 372, 370, 36]
['▁', '<0x48>', 'e', 'l', 'lo', '▁', '<0x57>', 'or', 'l', 'd', '<0x21>']


Llama 2 tokenizer proto If you'd like to export the raw protocol buffer for the tokenizer.model released by meta, this is the result:

normalizer_spec{

    name: "identity"
    precompiled_charsmap: ""
    add_dummy_prefix: true
    remove_extra_witespaces: false
    normalization_rule_tsv: ""

}

trainer_spec {

    input: "/large_experiments/theorem/datasets/MERGED/all.test1.merged"
    model_prefix: "spm_model_32k_200M_charcov099995_allowWSO__v2"
    model_type: BPE
    vocab_size: 32000
    self_test_sample_size: 0
    input_format: "text"
    character_coverage: 0.99995
    input_sentence_size: 200000000
    seed_sentencepiece_size: 1000000
    shrinking_factor: 0.75
    num_threads: 80
    num_sub_iterations: 2
    max_sentence_length: 4192
    shuffle_input_sentence: true
    max_sentencepiece_length: 16
    split_by_unicode_script: true
    split_by_whitespace: true
    split_by_number: true
    treat_whitespace_as_suffix: false
    split_digits: true
    allow_whitespace_only_pieces: true
    vocabulary_output_piece_score: true
    hard_vocab_limit: true
    use_all_vocab: false
    byte_fallback: true
    required_chars: ""
    unk_id: 0
    bos_id: 1
    eos_id: 2
    pad_id: -1
    unk_surface: "\342\201\207"
    unk_piece: "<unk>"
    bos_piece: "<s>"
    eos_piece: "</s>"
    pad_piece: "<pad>"
    train_extremely_large_corpus: false
    enable_differential_privacy: false
    differential_privacy_noise_level: 0.0
    differential_privacy_clipping_threshold: 0
}

##### Vocab_size
Vocab size influences the number of rows in embeddings and the number of output probabilities in the LM head
Note that small vocab size allow small sequences within a given context 
(transformer cannot see too much into the past), whereas large vocabulary size gives much more context to the model, this is good. But larger embedding layer and head lead to more parameters and could lead to LM being under trained, that is, onger sequences will lead to lower training example occurances and could lead to model under training. Also, as vocab size grows we are shrinking a lot of sequence to smaller context and the transformer do not get to understand each token as well as it wants to as too much information is compressed/squished to a token. vocab_size is a hyperparameter - sota it is 10K - 100K today.

How to set it and consideration around it
- Q: what should be the vocab size?
- Q: how can I increase vocab size?
- A: lets see. Reminder: gpt.py from before

##### Adding tokens - extending vocab size of a pre-trained model

This can be done. It is done for finetuning pre-trained model. For example adding a conversation agent, new tokens are added to maintain the metadata between the LM and the conversation. This is done with special tokens. Special tokens can also be added to using any other tool like the browser. This will need us to extend the size of embedding and the linear LM head, add special tokens to resize, freeze base model parameters and only train these new token parameters - mild model surgery.

There is an entire space of design to extend the vocab size that go way beyond adding special tokens.
Gist Tokens: Large prompts need to be encoded, attended to by the transformer, takes computation. So convert them to new tokens, put them in a sequence, train model by distillation for the new tokens only, obtain representation of new token (embeddings) while keeping the entire rest of the model frozen. Train such that behavior of the new model with new tokens is very similar to that obtained by utilizing long prompts. Compression technique for long prompts. At inference time obtain the new token representation and pass them to the base model to get the outputs. This is one of the techniques in class of parameter efficient fine tuning techniques. There is not training of base model or its weights - only prompt tokens representations are trained
Reference: Learning to compress prompts with Gist tokens https://arxiv.org/abs/2304.08467

With multimodality the convergence is that with a good scheme to convert the multimodalities to tokens (hard tokens where we force them to be integers or soft tokens where they are not forced to be discreet but go through bottlenecks - autoencoder), the training to predict the next token can be done using autoregressive models and hard tokens or diffusion models for soft tokens. 

SORA uses visual patches and pushes them through visual encoders, coverts images to tokens


##### Why are LLMs not good at spelling related tasks
How many 'l' is present in .DefaultCellStyle. GPT4 gives back 3 - there are 4. It is not able to see individual characters as it operates at the token level and if there are long compressed sequences in training it is not able to look at character level tasks well. Problem due to Tokenization approach. 

Needs explicit instructions like a) Step1: break the word to individual characters b) Step2: Count the number of 'l' or Reverse the sequence of characters. Numbers are represented completely arbitrarily in GPT-2 based on whatever happened to merge or not. Similarly does not recognize individual numbers - so poor arithmetic. Llama uses sentencepiece and splits up the digits

Both LLM and Tokenizers are not trained well on non-English data. The tokens are lot more diffused and shorter. Similarly, due to both training and tokenization issue GPT2 is not good with python - tokenization of spaces is not good in GPT2

Special tokens when input by the user are not recognized in chatGPT - text handling of user prompt vs special tokens
Trailing white space is added by LLM to the Payground (gpt instruct model - consider it closer to base model, that is sequence completer) response when it completes the sequence, but when space is added by user it is not understood by LM. (It is very rare that LM sees prompt ending with space, out of distribution)

Whenever it comes across partial tokens or spelling in input that are not seen - out of distribution. It emits arbitrary things (shocked - high perlexity). For some unstable tokens or trigger words the LM breaks and responds back hallucinatory,goes haywire or breaks safety guidelines even for benign prompts. It happens that the token was present in tokenization but not present in training so the embedding is initiated and never trained, so when triggered in inference it gives untrained behavior.

YAML is more efficient than JSON in terms or token economy with respect to density as it results to cost
